In [34]:
%pip install pandas requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [35]:
import pandas as pd
import requests
import re
import time
from pathlib import Path
from bs4 import BeautifulSoup

In [36]:
PROJECT_ROOT = Path("..")

FILINGS_DIR = PROJECT_ROOT / "data" / "raw" / "filings"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DASHBOARD_DIR = PROJECT_ROOT / "dashboard" / "data"

FILINGS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)

companies = {
    "Walmart": "0000104169",
    "Target": "0000027419",
    "Costco": "0000909832",
    "Home Depot": "0000354950",
    "Lowes": "0000060667"
}

headers = {
    "User-Agent": "Nisha Rajkumar nisha.rajkumar.offl@gmail.com"
}

In [37]:
# ============================================================
# GET LATEST 5 10-K FILINGS
# INCLUDING SEC ARCHIVED SUBMISSIONS
# ============================================================

def get_10k_filings(company, cik, number_of_filings=5):

    main_url = (
        f"https://data.sec.gov/submissions/CIK{cik}.json"
    )

    response = requests.get(
        main_url,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    # --------------------------------------------------------
    # CURRENT / RECENT FILINGS
    # --------------------------------------------------------

    all_submission_tables = []

    recent_filings = pd.DataFrame(
        data["filings"]["recent"]
    )

    all_submission_tables.append(
        recent_filings
    )

    # --------------------------------------------------------
    # OLDER ARCHIVED FILINGS
    # --------------------------------------------------------

    archive_files = data["filings"].get(
        "files",
        []
    )

    for archive in archive_files:

        archive_name = archive["name"]

        archive_url = (
            "https://data.sec.gov/submissions/"
            + archive_name
        )

        archive_response = requests.get(
            archive_url,
            headers=headers,
            timeout=30
        )

        archive_response.raise_for_status()

        archive_data = archive_response.json()

        archive_df = pd.DataFrame(
            archive_data
        )

        all_submission_tables.append(
            archive_df
        )

        time.sleep(0.2)

    # --------------------------------------------------------
    # COMBINE CURRENT + ARCHIVED
    # --------------------------------------------------------

    filings = pd.concat(
        all_submission_tables,
        ignore_index=True
    )

    filings = filings[
        filings["form"] == "10-K"
    ].copy()

    # Remove any duplicate filings
    filings = filings.drop_duplicates(
        subset=["accessionNumber"]
    )

    # Most recent first
    filings = filings.sort_values(
        "filingDate",
        ascending=False
    )

    filings = filings.head(
        number_of_filings
    )

    # --------------------------------------------------------
    # COMPANY INFORMATION
    # --------------------------------------------------------

    filings["company"] = company
    filings["cik"] = cik

    filings["accession_clean"] = (
        filings["accessionNumber"]
        .str.replace(
            "-",
            "",
            regex=False
        )
    )

    cik_clean = str(
        int(cik)
    )

    filings["filing_url"] = (
        "https://www.sec.gov/Archives/edgar/data/"
        + cik_clean
        + "/"
        + filings["accession_clean"]
        + "/"
        + filings["primaryDocument"]
    )

    return filings[
        [
            "company",
            "cik",
            "filingDate",
            "reportDate",
            "accessionNumber",
            "primaryDocument",
            "filing_url"
        ]
    ].reset_index(drop=True)

In [38]:
# ============================================================
# COLLECT 5 YEARS FOR ALL 5 COMPANIES
# ============================================================

all_filings = []

for company, cik in companies.items():

    company_filings = get_10k_filings(
        company,
        cik,
        number_of_filings=5
    )

    all_filings.append(
        company_filings
    )

    print(
        company,
        len(company_filings)
    )

    time.sleep(0.2)


filings_df = pd.concat(
    all_filings,
    ignore_index=True
)

filings_df["reportDate"] = pd.to_datetime(
    filings_df["reportDate"]
)

filings_df["fiscal_year"] = (
    filings_df["reportDate"].dt.year
)

print(
    "\nTOTAL FILINGS:",
    len(filings_df)
)

filings_df[
    [
        "company",
        "fiscal_year",
        "filingDate"
    ]
]

Walmart 5
Target 5
Costco 5
Home Depot 5
Lowes 5

TOTAL FILINGS: 25


,company,fiscal_year,filingDate
0,Walmart,2026,2026-03-13
1,Walmart,2025,2025-03-14
2,Walmart,2024,2024-03-15
3,Walmart,2023,2023-03-17
4,Walmart,2022,2022-03-18
5,Target,2026,2026-03-11
6,Target,2025,2025-03-12
7,Target,2024,2024-03-13
8,Target,2023,2023-03-08
9,Target,2022,2022-03-09


In [39]:
# ============================================================
# SAVE 10-K FILING METADATA
# ============================================================

filings_df["reportDate"] = pd.to_datetime(
    filings_df["reportDate"]
)

filings_df["fiscal_year"] = (
    filings_df["reportDate"].dt.year
)

filings_df.to_csv(
    PROCESSED_DIR / "filings_metadata.csv",
    index=False
)

print("Rows:", len(filings_df))
print(
    filings_df[
        ["company", "fiscal_year", "filing_url"]
    ].to_string(index=False)
)

Rows: 25
   company  fiscal_year                                                                          filing_url
   Walmart         2026  https://www.sec.gov/Archives/edgar/data/104169/000010416926000055/wmt-20260131.htm
   Walmart         2025  https://www.sec.gov/Archives/edgar/data/104169/000010416925000021/wmt-20250131.htm
   Walmart         2024  https://www.sec.gov/Archives/edgar/data/104169/000010416924000056/wmt-20240131.htm
   Walmart         2023  https://www.sec.gov/Archives/edgar/data/104169/000010416923000020/wmt-20230131.htm
   Walmart         2022  https://www.sec.gov/Archives/edgar/data/104169/000010416922000012/wmt-20220131.htm
    Target         2026   https://www.sec.gov/Archives/edgar/data/27419/000002741926000016/tgt-20260131.htm
    Target         2025   https://www.sec.gov/Archives/edgar/data/27419/000002741925000018/tgt-20250201.htm
    Target         2024   https://www.sec.gov/Archives/edgar/data/27419/000002741924000032/tgt-20240203.htm
    Target         

In [40]:
# ============================================================
# DOWNLOAD 10-K HTML FILINGS
# ============================================================

def download_filing(row):

    company_slug = (
        row["company"]
        .lower()
        .replace(" ", "_")
        .replace("'", "")
    )

    fiscal_year = int(row["fiscal_year"])

    file_path = (
        FILINGS_DIR /
        f"{company_slug}_{fiscal_year}.html"
    )

    # Skip if already downloaded
    if file_path.exists():
        return file_path

    response = requests.get(
        row["filing_url"],
        headers=headers,
        timeout=60
    )

    response.raise_for_status()

    file_path.write_text(
        response.text,
        encoding="utf-8"
    )

    time.sleep(0.2)

    return file_path


downloaded_paths = []

for _, row in filings_df.iterrows():

    try:

        path = download_filing(row)

        downloaded_paths.append(path)

        print(
            f'{row["company"]} '
            f'{row["fiscal_year"]}: downloaded'
        )

    except Exception as e:

        print(
            f'{row["company"]} '
            f'{row["fiscal_year"]}: FAILED -> {e}'
        )

Walmart 2026: downloaded
Walmart 2025: downloaded
Walmart 2024: downloaded
Walmart 2023: downloaded
Walmart 2022: downloaded
Target 2026: downloaded
Target 2025: downloaded
Target 2024: downloaded
Target 2023: downloaded
Target 2022: downloaded
Costco 2025: downloaded
Costco 2024: downloaded
Costco 2023: downloaded
Costco 2022: downloaded
Costco 2021: downloaded
Home Depot 2026: downloaded
Home Depot 2025: downloaded
Home Depot 2024: downloaded
Home Depot 2023: downloaded
Home Depot 2022: downloaded
Lowes 2026: downloaded
Lowes 2025: downloaded
Lowes 2024: downloaded
Lowes 2023: downloaded
Lowes 2022: downloaded


In [41]:
# ============================================================
# CLEAN SEC HTML
# ============================================================

def html_to_text(html):

    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    # Remove scripts and styles
    for tag in soup(
        ["script", "style", "noscript"]
    ):
        tag.decompose()

    text = soup.get_text(
        separator=" "
    )

    # Normalize whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [42]:
# ============================================================
# EXTRACT 10-K SECTIONS
# ============================================================

def extract_section(
    text,
    start_patterns,
    end_patterns,
    min_length=1000
):

    start_matches = []

    for pattern in start_patterns:

        start_matches.extend(
            list(
                re.finditer(
                    pattern,
                    text,
                    flags=re.IGNORECASE
                )
            )
        )

    end_matches = []

    for pattern in end_patterns:

        end_matches.extend(
            list(
                re.finditer(
                    pattern,
                    text,
                    flags=re.IGNORECASE
                )
            )
        )

    candidates = []

    for start in start_matches:

        for end in end_matches:

            if end.start() <= start.end():
                continue

            section = text[
                start.start():
                end.start()
            ].strip()

            if len(section) >= min_length:

                candidates.append(
                    section
                )

    if not candidates:
        return None

    # Usually the actual section is much larger
    # than the table-of-contents reference
    return max(
        candidates,
        key=len
    )

In [43]:
# ============================================================
# SECTION HEADING PATTERNS
# ============================================================

risk_start_patterns = [
    r"\bitem\s+1a[\.\:\-\s]+risk\s+factors\b",
    r"\bitem\s+1a\b.{0,50}\brisk\s+factors\b"
]

risk_end_patterns = [
    r"\bitem\s+1b\b",
    r"\bitem\s+1c\b",
    r"\bitem\s+2\b"
]


mda_start_patterns = [
    r"\bitem\s+7[\.\:\-\s]+management.{0,100}discussion.{0,100}analysis\b",
    r"\bitem\s+7\b.{0,150}\bmanagement\b.{0,150}\bdiscussion\b"
]

mda_end_patterns = [
    r"\bitem\s+7a\b",
    r"\bitem\s+8\b"
]

In [44]:
# ============================================================
# EXTRACT RISK FACTORS + MD&A
# ============================================================

section_rows = []

for _, row in filings_df.iterrows():

    company_slug = (
        row["company"]
        .lower()
        .replace(" ", "_")
        .replace("'", "")
    )

    fiscal_year = int(
        row["fiscal_year"]
    )

    file_path = (
        FILINGS_DIR /
        f"{company_slug}_{fiscal_year}.html"
    )

    if not file_path.exists():
        continue

    html = file_path.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    clean_text = html_to_text(html)

    risk_text = extract_section(
        clean_text,
        risk_start_patterns,
        risk_end_patterns
    )

    mda_text = extract_section(
        clean_text,
        mda_start_patterns,
        mda_end_patterns
    )

    section_rows.append({
        "company": row["company"],
        "fiscal_year": fiscal_year,
        "filing_date": row["filingDate"],
        "risk_factors": risk_text,
        "mda": mda_text
    })


sections_df = pd.DataFrame(
    section_rows
)

print(
    "Filings processed:",
    len(sections_df)
)

sections_df.head()

Filings processed: 25


,company,fiscal_year,filing_date,risk_factors,mda
0,Walmart,2026,2026-03-13,Item 1A Risk Factors 13 Item 1B Unresolved Sta...,Item 7 Management's Discussion and Analysis of...
1,Walmart,2025,2025-03-14,Item 1A Risk Factors 14 Item 1B Unresolved Sta...,Item 7 Management's Discussion and Analysis of...
2,Walmart,2024,2024-03-15,Item 1A Risk Factors 15 Item 1B Unresolved Sta...,Item 7 Management's Discussion and Analysis of...
3,Walmart,2023,2023-03-17,Item 1A Risk Factors 15 Item 1B Unresolved Sta...,Item 7 Management's Discussion and Analysis of...
4,Walmart,2022,2022-03-18,Item 1A Risk Factors 15 Item 1B Unresolved Sta...,Item 7 Management's Discussion and Analysis of...


In [45]:
# ============================================================
# SECTION EXTRACTION QUALITY CHECK
# ============================================================

sections_df["risk_chars"] = (
    sections_df["risk_factors"]
    .fillna("")
    .str.len()
)

sections_df["mda_chars"] = (
    sections_df["mda"]
    .fillna("")
    .str.len()
)

quality_check = sections_df[
    [
        "company",
        "fiscal_year",
        "risk_chars",
        "mda_chars"
    ]
].copy()

quality_check

,company,fiscal_year,risk_chars,mda_chars
0,Walmart,2026,148451,340760
1,Walmart,2025,154969,352734
2,Walmart,2024,152076,357740
3,Walmart,2023,133915,340122
4,Walmart,2022,127238,329729
5,Target,2026,84093,213376
6,Target,2025,76706,204023
7,Target,2024,59930,186096
8,Target,2023,47051,174933
9,Target,2022,47506,179867


In [46]:
print(
    "Missing Risk Factors:",
    sections_df["risk_factors"]
    .isna()
    .sum()
)

print(
    "Missing MD&A:",
    sections_df["mda"]
    .isna()
    .sum()
)

Missing Risk Factors: 0
Missing MD&A: 0


In [47]:
# ============================================================
# SAVE EXTRACTED 10-K TEXT
# ============================================================

sections_df.to_csv(
    PROCESSED_DIR /
    "management_risk_sections.csv",
    index=False
)

print(
    "Saved:",
    PROCESSED_DIR /
    "management_risk_sections.csv"
)

Saved: ..\data\processed\management_risk_sections.csv


In [48]:
# ============================================================
# RISK TOPIC DICTIONARY
# ============================================================

risk_topics = {

    "Consumer Demand": [
        "consumer demand",
        "consumer spending",
        "customer demand",
        "discretionary spending",
        "customer traffic"
    ],

    "Inflation & Costs": [
        "inflation",
        "cost inflation",
        "input costs",
        "commodity costs",
        "freight costs",
        "higher costs"
    ],

    "Inventory": [
        "inventory",
        "inventories",
        "markdown",
        "shrink",
        "stock levels"
    ],

    "Supply Chain": [
        "supply chain",
        "logistics",
        "distribution",
        "transportation",
        "sourcing",
        "suppliers"
    ],

    "Labor": [
        "labor",
        "wages",
        "workforce",
        "employee costs",
        "compensation",
        "staffing"
    ],

    "Debt & Interest Rates": [
        "debt",
        "borrowings",
        "interest rates",
        "financing costs",
        "credit facility",
        "credit facilities"
    ],

    "Capital Investment": [
        "capital expenditures",
        "capital expenditure",
        "capex",
        "store investment",
        "technology investment"
    ],

    "Competition": [
        "competition",
        "competitive",
        "competitors",
        "market share"
    ],

    "Cybersecurity": [
        "cybersecurity",
        "cyber security",
        "cyberattack",
        "cyber attack",
        "data breach",
        "information security"
    ],

    "Economic Uncertainty": [
        "economic uncertainty",
        "macroeconomic",
        "economic conditions",
        "recession",
        "unemployment",
        "geopolitical"
    ],

    "Regulation": [
        "regulation",
        "regulatory",
        "compliance",
        "laws and regulations"
    ]
}

In [49]:
# ============================================================
# NLP HELPER FUNCTIONS
# ============================================================

def word_count(text):

    if not text:
        return 0

    return len(
        re.findall(
            r"\b[a-zA-Z]+\b",
            text
        )
    )


def count_term(text, term):

    if not text:
        return 0

    pattern = (
        r"\b" +
        re.escape(term) +
        r"\b"
    )

    return len(
        re.findall(
            pattern,
            text,
            flags=re.IGNORECASE
        )
    )


def count_topic(text, keywords):

    return sum(
        count_term(
            text,
            keyword
        )
        for keyword in keywords
    )

In [50]:
# ============================================================
# BUILD RISK TOPIC DATASET
# ============================================================

topic_rows = []

for _, row in sections_df.iterrows():

    for section_name, text_column in [
        ("Risk Factors", "risk_factors"),
        ("MD&A", "mda")
    ]:

        text = row[text_column]

        words = word_count(text)

        for topic, keywords in risk_topics.items():

            mentions = count_topic(
                text,
                keywords
            )

            mentions_per_10k = (
                mentions / words * 10000
                if words > 0
                else 0
            )

            topic_rows.append({
                "company": row["company"],
                "fiscal_year": row["fiscal_year"],
                "section": section_name,
                "topic": topic,
                "mentions": mentions,
                "word_count": words,
                "mentions_per_10k_words":
                    mentions_per_10k
            })


risk_topics_df = pd.DataFrame(
    topic_rows
)

risk_topics_df.head(20)

,company,fiscal_year,section,topic,mentions,word_count,mentions_per_10k_words
0,Walmart,2026,Risk Factors,Consumer Demand,8,21703,3.686126
1,Walmart,2026,Risk Factors,Inflation & Costs,6,21703,2.764595
2,Walmart,2026,Risk Factors,Inventory,14,21703,6.450721
3,Walmart,2026,Risk Factors,Supply Chain,94,21703,43.311985
4,Walmart,2026,Risk Factors,Labor,38,21703,17.509100
5,Walmart,2026,Risk Factors,Debt & Interest Rates,4,21703,1.843063
6,Walmart,2026,Risk Factors,Capital Investment,5,21703,2.303829
7,Walmart,2026,Risk Factors,Competition,52,21703,23.959821
8,Walmart,2026,Risk Factors,Cybersecurity,72,21703,33.175137
9,Walmart,2026,Risk Factors,Economic Uncertainty,11,21703,5.068424


In [51]:
# ============================================================
# YEAR-OVER-YEAR RISK TOPIC CHANGE
# ============================================================

risk_topics_df = (
    risk_topics_df
    .sort_values(
        [
            "company",
            "section",
            "topic",
            "fiscal_year"
        ]
    )
    .reset_index(drop=True)
)

risk_topics_df[
    "topic_rate_change"
] = (
    risk_topics_df
    .groupby(
        [
            "company",
            "section",
            "topic"
        ]
    )[
        "mentions_per_10k_words"
    ]
    .diff()
)

risk_topics_df.head()

,company,fiscal_year,section,topic,mentions,word_count,mentions_per_10k_words,topic_rate_change
0,Costco,2021,MD&A,Capital Investment,4,28239,1.416481,NaN
1,Costco,2022,MD&A,Capital Investment,6,27132,2.211411,0.794930
2,Costco,2023,MD&A,Capital Investment,6,26775,2.240896,0.029485
3,Costco,2024,MD&A,Capital Investment,6,28001,2.142781,-0.098116
4,Costco,2025,MD&A,Capital Investment,7,27624,2.534028,0.391248


In [52]:
# ============================================================
# MANAGEMENT / RISK LANGUAGE SIGNALS
# ============================================================

uncertainty_terms = [
    "uncertain",
    "uncertainty",
    "uncertainties",
    "volatile",
    "volatility",
    "unpredictable",
    "risk",
    "risks",
    "adverse",
    "disruption",
    "disruptions",
    "pressure",
    "pressures",
    "challenging",
    "challenges"
]


negative_terms = [
    "decline",
    "declined",
    "decrease",
    "decreased",
    "weakness",
    "weak",
    "loss",
    "losses",
    "higher costs",
    "adverse",
    "deterioration",
    "lower demand"
]


positive_terms = [
    "growth",
    "improved",
    "improvement",
    "strong",
    "increase",
    "increased",
    "opportunity",
    "opportunities",
    "benefit",
    "benefits",
    "resilient"
]

In [53]:
# ============================================================
# BUILD MANAGEMENT LANGUAGE SIGNAL DATASET
# ============================================================

signal_rows = []

for _, row in sections_df.iterrows():

    for section_name, text_column in [
        ("Risk Factors", "risk_factors"),
        ("MD&A", "mda")
    ]:

        text = row[text_column]

        words = word_count(text)

        uncertainty_count = sum(
            count_term(text, term)
            for term in uncertainty_terms
        )

        negative_count = sum(
            count_term(text, term)
            for term in negative_terms
        )

        positive_count = sum(
            count_term(text, term)
            for term in positive_terms
        )

        signal_rows.append({

            "company":
                row["company"],

            "fiscal_year":
                row["fiscal_year"],

            "section":
                section_name,

            "word_count":
                words,

            "uncertainty_mentions":
                uncertainty_count,

            "negative_mentions":
                negative_count,

            "positive_mentions":
                positive_count,

            "uncertainty_per_10k_words":
                uncertainty_count / words * 10000
                if words > 0 else 0,

            "negative_per_10k_words":
                negative_count / words * 10000
                if words > 0 else 0,

            "positive_per_10k_words":
                positive_count / words * 10000
                if words > 0 else 0
        })


risk_signals_df = pd.DataFrame(
    signal_rows
)

risk_signals_df.head()

,company,fiscal_year,section,word_count,uncertainty_mentions,negative_mentions,positive_mentions,uncertainty_per_10k_words,negative_per_10k_words,positive_per_10k_words
0,Walmart,2026,Risk Factors,21703,141,54,95,64.967977,24.881353,43.772750
1,Walmart,2026,MD&A,48068,188,166,282,39.111259,34.534410,58.666889
2,Walmart,2025,Risk Factors,22812,146,56,103,64.001403,24.548483,45.151675
3,Walmart,2025,MD&A,49935,196,179,269,39.251026,35.846601,53.870031
4,Walmart,2024,Risk Factors,22467,136,50,99,60.533227,22.254863,44.064628


In [54]:
# ============================================================
# BUILD MANAGEMENT LANGUAGE SIGNAL DATASET
# ============================================================

signal_rows = []

for _, row in sections_df.iterrows():

    for section_name, text_column in [
        ("Risk Factors", "risk_factors"),
        ("MD&A", "mda")
    ]:

        text = row[text_column]

        words = word_count(text)

        uncertainty_count = sum(
            count_term(text, term)
            for term in uncertainty_terms
        )

        negative_count = sum(
            count_term(text, term)
            for term in negative_terms
        )

        positive_count = sum(
            count_term(text, term)
            for term in positive_terms
        )

        signal_rows.append({

            "company":
                row["company"],

            "fiscal_year":
                row["fiscal_year"],

            "section":
                section_name,

            "word_count":
                words,

            "uncertainty_mentions":
                uncertainty_count,

            "negative_mentions":
                negative_count,

            "positive_mentions":
                positive_count,

            "uncertainty_per_10k_words":
                uncertainty_count / words * 10000
                if words > 0 else 0,

            "negative_per_10k_words":
                negative_count / words * 10000
                if words > 0 else 0,

            "positive_per_10k_words":
                positive_count / words * 10000
                if words > 0 else 0
        })


risk_signals_df = pd.DataFrame(
    signal_rows
)

risk_signals_df.head()

,company,fiscal_year,section,word_count,uncertainty_mentions,negative_mentions,positive_mentions,uncertainty_per_10k_words,negative_per_10k_words,positive_per_10k_words
0,Walmart,2026,Risk Factors,21703,141,54,95,64.967977,24.881353,43.772750
1,Walmart,2026,MD&A,48068,188,166,282,39.111259,34.534410,58.666889
2,Walmart,2025,Risk Factors,22812,146,56,103,64.001403,24.548483,45.151675
3,Walmart,2025,MD&A,49935,196,179,269,39.251026,35.846601,53.870031
4,Walmart,2024,Risk Factors,22467,136,50,99,60.533227,22.254863,44.064628


In [55]:
# ============================================================
# RISK LANGUAGE TREND CHANGE
# ============================================================

risk_signals_df = (
    risk_signals_df
    .sort_values(
        [
            "company",
            "section",
            "fiscal_year"
        ]
    )
    .reset_index(drop=True)
)

for metric in [
    "uncertainty_per_10k_words",
    "negative_per_10k_words",
    "positive_per_10k_words"
]:

    risk_signals_df[
        f"{metric}_change"
    ] = (
        risk_signals_df
        .groupby(
            [
                "company",
                "section"
            ]
        )[metric]
        .diff()
    )

risk_signals_df.head()

,company,fiscal_year,section,word_count,uncertainty_mentions,negative_mentions,positive_mentions,uncertainty_per_10k_words,negative_per_10k_words,positive_per_10k_words,uncertainty_per_10k_words_change,negative_per_10k_words_change,positive_per_10k_words_change
0,Costco,2021,MD&A,28239,111,92,138,39.307341,32.579057,48.868586,NaN,NaN,NaN
1,Costco,2022,MD&A,27132,101,92,128,37.225416,33.908300,47.176765,-2.081924,1.329243,-1.691821
2,Costco,2023,MD&A,26775,95,84,127,35.480859,31.372549,47.432306,-1.744557,-2.535751,0.255541
3,Costco,2024,MD&A,28001,113,84,136,40.355702,29.998929,48.569694,4.874843,-1.373620,1.137388
4,Costco,2025,MD&A,27624,115,79,120,41.630466,28.598320,43.440487,1.274765,-1.400608,-5.129207


In [56]:
# ============================================================
# LATEST RISK TOPICS BY COMPANY
# ============================================================

latest_years = (
    risk_topics_df
    .groupby("company")[
        "fiscal_year"
    ]
    .max()
    .reset_index()
    .rename(
        columns={
            "fiscal_year":
            "latest_year"
        }
    )
)

latest_topics = (
    risk_topics_df
    .merge(
        latest_years,
        on="company"
    )
)

latest_topics = latest_topics[
    latest_topics["fiscal_year"]
    ==
    latest_topics["latest_year"]
].copy()


latest_topics[
    [
        "company",
        "section",
        "topic",
        "mentions_per_10k_words"
    ]
].sort_values(
    [
        "company",
        "section",
        "mentions_per_10k_words"
    ],
    ascending=[
        True,
        True,
        False
    ]
).head(50)

,company,section,topic,mentions_per_10k_words
24,Costco,MD&A,Debt & Interest Rates,23.530264
44,Costco,MD&A,Labor,20.996235
54,Costco,MD&A,Supply Chain,15.204170
39,Costco,MD&A,Inventory,13.756154
49,Costco,MD&A,Regulation,13.756154
9,Costco,MD&A,Competition,11.584130
19,Costco,MD&A,Cybersecurity,11.222126
29,Costco,MD&A,Economic Uncertainty,3.258036
4,Costco,MD&A,Capital Investment,2.534028
34,Costco,MD&A,Inflation & Costs,2.534028


In [57]:
# ============================================================
# SAVE NLP OUTPUTS
# ============================================================

risk_topics_df.to_csv(
    PROCESSED_DIR /
    "risk_topics.csv",
    index=False
)

risk_signals_df.to_csv(
    PROCESSED_DIR /
    "risk_signals.csv",
    index=False
)

latest_topics.to_csv(
    PROCESSED_DIR /
    "latest_risk_topics.csv",
    index=False
)


print(
    "✅ Saved risk_topics.csv"
)

print(
    "✅ Saved risk_signals.csv"
)

print(
    "✅ Saved latest_risk_topics.csv"
)

✅ Saved risk_topics.csv
✅ Saved risk_signals.csv
✅ Saved latest_risk_topics.csv


In [58]:
# ============================================================
# EXPORT NLP DATA FOR POWER BI
# ============================================================

dashboard_risk_topics = (
    risk_topics_df.copy()
)

dashboard_risk_signals = (
    risk_signals_df.copy()
)

dashboard_latest_topics = (
    latest_topics.copy()
)


for df in [
    dashboard_risk_topics,
    dashboard_risk_signals,
    dashboard_latest_topics
]:

    df["company"] = (
        df["company"]
        .replace({
            "Lowes":
            "Lowe's"
        })
    )


dashboard_risk_topics.to_csv(
    DASHBOARD_DIR /
    "risk_topics.csv",
    index=False
)

dashboard_risk_signals.to_csv(
    DASHBOARD_DIR /
    "risk_signals.csv",
    index=False
)

dashboard_latest_topics.to_csv(
    DASHBOARD_DIR /
    "latest_risk_topics.csv",
    index=False
)


print(
    "✅ Power BI NLP datasets created"
)

✅ Power BI NLP datasets created


In [59]:
# ============================================================
# FINAL NLP QUALITY CHECK
# ============================================================

print(
    "Filings analyzed:",
    len(sections_df)
)

print(
    "Risk topic rows:",
    len(risk_topics_df)
)

print(
    "Risk signal rows:",
    len(risk_signals_df)
)

print(
    "\nMissing Risk Factors:",
    sections_df["risk_factors"]
    .isna()
    .sum()
)

print(
    "Missing MD&A:",
    sections_df["mda"]
    .isna()
    .sum()
)

print(
    "\nDashboard files:"
)

for file in DASHBOARD_DIR.glob(
    "*risk*.csv"
):
    print(file.name)

Filings analyzed: 25
Risk topic rows: 550
Risk signal rows: 50

Missing Risk Factors: 0
Missing MD&A: 0

Dashboard files:
latest_risk_topics.csv
risk_signals.csv
risk_topics.csv


In [60]:
# ============================================================
# DIAGNOSE MISSING FILINGS + DASHBOARD FILES
# ============================================================

expected = filings_df[
    ["company", "fiscal_year"]
].copy()

processed = sections_df[
    ["company", "fiscal_year"]
].copy()

missing_filings = expected.merge(
    processed,
    on=["company", "fiscal_year"],
    how="left",
    indicator=True
)

missing_filings = missing_filings[
    missing_filings["_merge"] == "left_only"
][
    ["company", "fiscal_year"]
]

print("EXPECTED FILINGS:", len(expected))
print("PROCESSED FILINGS:", len(processed))

print("\nMISSING FILINGS:")
print(
    missing_filings.to_string(index=False)
)


print("\nFILES ACTUALLY DOWNLOADED:")

for file in sorted(FILINGS_DIR.glob("*.html")):
    print(file.name)


print("\nDASHBOARD NLP FILE CHECK:")

files_to_check = [
    "risk_topics.csv",
    "risk_signals.csv",
    "latest_risk_topics.csv"
]

for filename in files_to_check:

    path = DASHBOARD_DIR / filename

    print(
        filename,
        "->",
        "FOUND" if path.exists() else "MISSING"
    )

EXPECTED FILINGS: 25
PROCESSED FILINGS: 25

MISSING FILINGS:
Empty DataFrame
Columns: [company, fiscal_year]
Index: []

FILES ACTUALLY DOWNLOADED:
costco_2021.html
costco_2022.html
costco_2023.html
costco_2024.html
costco_2025.html
home_depot_2022.html
home_depot_2023.html
home_depot_2024.html
home_depot_2025.html
home_depot_2026.html
lowes_2022.html
lowes_2023.html
lowes_2024.html
lowes_2025.html
lowes_2026.html
target_2022.html
target_2023.html
target_2024.html
target_2025.html
target_2026.html
walmart_2022.html
walmart_2023.html
walmart_2024.html
walmart_2025.html
walmart_2026.html

DASHBOARD NLP FILE CHECK:
risk_topics.csv -> FOUND
risk_signals.csv -> FOUND
latest_risk_topics.csv -> FOUND


In [61]:
# ============================================================
# VALIDATE LATEST RISK TOPICS
# ============================================================

latest_risk_view = (
    latest_topics[
        [
            "company",
            "section",
            "topic",
            "mentions",
            "mentions_per_10k_words"
        ]
    ]
    .sort_values(
        [
            "company",
            "section",
            "mentions_per_10k_words"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)

for company in latest_risk_view["company"].unique():

    print("\n" + "=" * 60)
    print(company)
    print("=" * 60)

    company_data = latest_risk_view[
        latest_risk_view["company"] == company
    ]

    for section in company_data["section"].unique():

        print(f"\n{section}")

        display(
            company_data[
                company_data["section"] == section
            ].head(5)
        )


Costco

MD&A


,company,section,topic,mentions,mentions_per_10k_words
24,Costco,MD&A,Debt & Interest Rates,65,23.530264
44,Costco,MD&A,Labor,58,20.996235
54,Costco,MD&A,Supply Chain,42,15.204170
39,Costco,MD&A,Inventory,38,13.756154
49,Costco,MD&A,Regulation,38,13.756154



Risk Factors


,company,section,topic,mentions,mentions_per_10k_words
104,Costco,Risk Factors,Regulation,33,31.853282
74,Costco,Risk Factors,Cybersecurity,32,30.888031
109,Costco,Risk Factors,Supply Chain,32,30.888031
64,Costco,Risk Factors,Competition,25,24.131274
99,Costco,Risk Factors,Labor,19,18.339768



Home Depot

MD&A


...

,company,section,topic,mentions,mentions_per_10k_words
404,Target,Risk Factors,Cybersecurity,65,53.843605
439,Target,Risk Factors,Supply Chain,44,36.447979
429,Target,Risk Factors,Labor,40,33.134526
434,Target,Risk Factors,Regulation,35,28.992710
394,Target,Risk Factors,Competition,25,20.709079



Walmart

MD&A


,company,section,topic,mentions,mentions_per_10k_words
494,Walmart,MD&A,Supply Chain,132,27.461097
464,Walmart,MD&A,Debt & Interest Rates,113,23.508363
489,Walmart,MD&A,Regulation,98,20.387784
459,Walmart,MD&A,Cybersecurity,72,14.978780
484,Walmart,MD&A,Labor,72,14.978780



Risk Factors


,company,section,topic,mentions,mentions_per_10k_words
549,Walmart,Risk Factors,Supply Chain,94,43.311985
544,Walmart,Risk Factors,Regulation,85,39.165092
514,Walmart,Risk Factors,Cybersecurity,72,33.175137
504,Walmart,Risk Factors,Competition,52,23.959821
539,Walmart,Risk Factors,Labor,38,17.509100


In [62]:
# ============================================================
# VALIDATE MANAGEMENT LANGUAGE SIGNALS
# ============================================================

latest_signal_year = (
    risk_signals_df
    .groupby("company")["fiscal_year"]
    .max()
    .reset_index()
)

latest_signals = risk_signals_df.merge(
    latest_signal_year,
    on=["company", "fiscal_year"]
)

latest_signals[
    [
        "company",
        "section",
        "fiscal_year",
        "uncertainty_per_10k_words",
        "negative_per_10k_words",
        "positive_per_10k_words"
    ]
].sort_values(
    [
        "section",
        "uncertainty_per_10k_words"
    ],
    ascending=[
        True,
        False
    ]
)

,company,section,fiscal_year,uncertainty_per_10k_words,negative_per_10k_words,positive_per_10k_words
6,Target,MD&A,2026,51.247011,28.356679,49.538777
4,Lowes,MD&A,2026,42.314638,21.022559,49.861197
2,Home Depot,MD&A,2026,42.275085,20.337744,40.675487
0,Costco,MD&A,2025,41.630466,28.598320,43.440487
8,Walmart,MD&A,2026,39.111259,34.534410,58.666889
7,Target,Risk Factors,2026,87.806494,18.223989,47.216700
5,Lowes,Risk Factors,2026,80.701266,20.871017,76.527063
1,Costco,Risk Factors,2025,80.115830,31.853282,41.505792
3,Home Depot,Risk Factors,2026,72.211410,17.412681,55.310868
9,Walmart,Risk Factors,2026,64.967977,24.881353,43.772750


In [64]:
# ============================================================
# CREATE LATEST MANAGEMENT RISK SIGNALS FOR POWER BI
# ============================================================

latest_signal_years = (
    risk_signals_df
    .groupby("company")["fiscal_year"]
    .max()
    .reset_index()
    .rename(
        columns={
            "fiscal_year": "latest_year"
        }
    )
)

latest_risk_signals = (
    risk_signals_df
    .merge(
        latest_signal_years,
        on="company",
        how="left"
    )
)

latest_risk_signals = latest_risk_signals[
    latest_risk_signals["fiscal_year"]
    ==
    latest_risk_signals["latest_year"]
].copy()

latest_risk_signals["company"] = (
    latest_risk_signals["company"]
    .replace({
        "Lowes": "Lowe's"
    })
)

latest_risk_signals.to_csv(
    DASHBOARD_DIR /
    "latest_risk_signals.csv",
    index=False
)

print(
    "✅ latest_risk_signals.csv created"
)

print(
    "Rows:",
    len(latest_risk_signals)
)

latest_risk_signals[
    [
        "company",
        "fiscal_year",
        "section",
        "uncertainty_per_10k_words",
        "negative_per_10k_words",
        "positive_per_10k_words"
    ]
]

✅ latest_risk_signals.csv created
Rows: 10


,company,fiscal_year,section,uncertainty_per_10k_words,negative_per_10k_words,positive_per_10k_words
4,Costco,2025,MD&A,41.630466,28.598320,43.440487
9,Costco,2025,Risk Factors,80.115830,31.853282,41.505792
14,Home Depot,2026,MD&A,42.275085,20.337744,40.675487
19,Home Depot,2026,Risk Factors,72.211410,17.412681,55.310868
24,Lowe's,2026,MD&A,42.314638,21.022559,49.861197
29,Lowe's,2026,Risk Factors,80.701266,20.871017,76.527063
34,Target,2026,MD&A,51.247011,28.356679,49.538777
39,Target,2026,Risk Factors,87.806494,18.223989,47.216700
44,Walmart,2026,MD&A,39.111259,34.534410,58.666889
49,Walmart,2026,Risk Factors,64.967977,24.881353,43.772750


In [65]:
# ============================================================
# CHECK LATEST RISK SIGNALS FILE LOCATION
# ============================================================

latest_file = (
    DASHBOARD_DIR /
    "latest_risk_signals.csv"
)

print("Expected location:")
print(latest_file.resolve())

print("\nFile exists:")
print(latest_file.exists())

Expected location:
C:\Users\Nisha Rajkumar\OneDrive\Desktop\projects\Corporate_Credit_Analysis\dashboard\data\latest_risk_signals.csv

File exists:
True
